In [ ]:
!pip install mediapipe opencv-python scipy pygame

Note: you may need to restart the kernel to use updated packages.Collecting mediapipe
     ---------------------------------------- 10.9/10.9 MB 3.7 MB/s eta 0:00:00
     ---------------------------------------- 40.2/40.2 MB 3.1 MB/s eta 0:00:00
     ---------------------------------------- 10.6/10.6 MB 3.1 MB/s eta 0:00:00
     -------------------------------------- 365.3/365.3 KB 2.3 MB/s eta 0:00:00
     ---------------------------------------- 46.5/46.5 MB 2.4 MB/s eta 0:00:00
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl (26 kB)
  Using cached absl_py-2.3.1-py3-none-any.whl (135 kB)
     -------------------------------------- 182.8/182.8 KB 2.8 MB/s eta 0:00:00
     -------------------------------------- 118.1/118.1 KB 2.3 MB/s eta 0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.2.10
    Uninstalling flatbuffers-25.2.10:
      Successfully uninstalled flatbuffers-25.2.10
  Attempting uninstall: absl-py
    Found existing insta

You should consider upgrading via the 'c:\Users\Maha\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip' command.


In [2]:
"""
==============================================================
  Driver Drowsiness Detector  (OpenCV-only edition)
  Uses: OpenCV Haar cascades — no external model downloads
  Run:  python drowsiness_detector.py

SETUP (run once):
    pip install opencv-python scipy pygame

HOW IT WORKS:
  - Detects face + eyes with built-in Haar cascades
  - Eye Aspect Ratio (EAR) from eye bounding-box height/width
  - Mouth/smile cascade for yawn detection
  - PERCLOS rolling window (% of time eyes closed)
  - Head-nod proxy: face bounding-box vertical shift
  - Audio + visual alarm after sustained drowsiness

CONTROLS:
  Q        → quit
  R        → reset all counters
  S        → save snapshot
  +/-      → raise / lower EAR sensitivity
==============================================================
"""

import cv2
import numpy as np
import time
import sys
from datetime import datetime
from collections import deque

# ── optional audio ────────────────────────────────────────────────────────────
AUDIO_OK = False
try:
    import pygame
    pygame.mixer.init(frequency=44100, size=-16, channels=1, buffer=512)
    AUDIO_OK = True
except Exception:
    pass

# ── cascade paths ─────────────────────────────────────────────────────────────
DATA = cv2.data.haarcascades
FACE_CASCADE  = cv2.CascadeClassifier(DATA + "haarcascade_frontalface_default.xml")
EYE_CASCADE   = cv2.CascadeClassifier(DATA + "haarcascade_eye.xml")
SMILE_CASCADE = cv2.CascadeClassifier(DATA + "haarcascade_smile.xml")

# ── thresholds ────────────────────────────────────────────────────────────────
EAR_THRESH        = 0.22   # eye h/w ratio below this → closed
EAR_CONSEC_FRAMES = 20     # consecutive closed frames → alarm
PERCLOS_WINDOW    = 150    # rolling window in frames
PERCLOS_ALERT     = 0.35   # PERCLOS fraction → drowsy
HEAD_NOD_THRESH   = 0.08   # face-centre vertical drop (ratio of face height)
YAWN_CONSEC       = 18     # smile-detected frames to count as yawn

# ── state ─────────────────────────────────────────────────────────────────────
ear_counter    = 0
yawn_counter   = 0
alarm_on       = False
session_yawns  = 0
session_blinks = 0
blink_open     = True
perclos_buf    = deque(maxlen=PERCLOS_WINDOW)
face_y_history = deque(maxlen=40)   # face centre-y for nod detection
fps_deque      = deque(maxlen=20)

# ── colours (BGR) ─────────────────────────────────────────────────────────────
CLR_OK    = (50,  220, 80)
CLR_WARN  = (0,   200, 255)
CLR_ALARM = (0,   30,  240)
CLR_TEXT  = (220, 220, 220)
CLR_PANEL = (35,  35,  35)

# ── audio ─────────────────────────────────────────────────────────────────────
def make_beep(freq=880, duration=0.5, vol=0.7):
    if not AUDIO_OK:
        return None
    sr = 44100
    t  = np.linspace(0, duration, int(sr * duration), endpoint=False)
    wave   = (np.sin(2 * np.pi * freq * t) * vol * 32767).astype(np.int16)
    stereo = np.column_stack([wave, wave])
    return pygame.sndarray.make_sound(stereo)

beep_sound = make_beep()

def play_alarm():
    if AUDIO_OK and beep_sound and not pygame.mixer.get_busy():
        beep_sound.play(-1)

def stop_alarm():
    if AUDIO_OK:
        pygame.mixer.stop()

# ── EAR from bounding box ─────────────────────────────────────────────────────
def box_ear(ew, eh):
    """
    Approximate EAR from Haar eye bounding box.
    Open eye  → taller box  → higher ratio.
    Closed eye → flatter box → lower ratio.
    Normalised so 'neutral open' ≈ 0.30-0.40.
    """
    return (eh / (ew + 1e-6)) * 0.70   # scale factor calibrates to ~0.22 threshold

# ── HUD helpers ───────────────────────────────────────────────────────────────
def draw_panel(frame, x, y, w, h, alpha=0.55):
    ov = frame.copy()
    cv2.rectangle(ov, (x, y), (x + w, y + h), CLR_PANEL, -1)
    cv2.addWeighted(ov, alpha, frame, 1 - alpha, 0, frame)

def draw_gauge(frame, x, y, label, value, lo, hi, thresh, color, bar_w=120, bar_h=14):
    ratio  = min(max((value - lo) / (hi - lo + 1e-6), 0), 1)
    filled = int(bar_w * ratio)
    tpx    = int(bar_w * (thresh - lo) / (hi - lo + 1e-6))
    cv2.rectangle(frame, (x, y), (x + bar_w, y + bar_h), (60, 60, 60), -1)
    cv2.rectangle(frame, (x, y), (x + filled, y + bar_h), color, -1)
    cv2.line(frame, (x + tpx, y - 2), (x + tpx, y + bar_h + 2), (255, 255, 100), 1)
    cv2.putText(frame, f"{label}: {value:.2f}",
                (x, y - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.42, CLR_TEXT, 1, cv2.LINE_AA)

def put(frame, text, pos, scale=0.55, color=CLR_TEXT, thickness=1):
    cv2.putText(frame, text, pos,
                cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness, cv2.LINE_AA)

def alert_overlay(frame, message, color):
    H, W = frame.shape[:2]
    ov = frame.copy()
    cv2.rectangle(ov, (0, H - 80), (W, H), color, -1)
    cv2.addWeighted(ov, 0.6, frame, 0.4, 0, frame)
    (tw, _), _ = cv2.getTextSize(message, cv2.FONT_HERSHEY_DUPLEX, 1.1, 2)
    cv2.putText(frame, message,
                ((W - tw) // 2, H - 24),
                cv2.FONT_HERSHEY_DUPLEX, 1.1, (255, 255, 255), 2, cv2.LINE_AA)

# ── main ──────────────────────────────────────────────────────────────────────
def main():
    global ear_counter, yawn_counter, alarm_on
    global session_yawns, session_blinks, blink_open
    global EAR_THRESH

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("[✗] Cannot open webcam.")
        sys.exit(1)

    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    print("\n[▶] Drowsiness Detector running (OpenCV cascade edition)")
    print("    Q=quit  R=reset  S=save snapshot  +/-=sensitivity\n")
    if not AUDIO_OK:
        print("    [!] pygame not found — no audio alarm. pip install pygame\n")

    t_prev = time.time()

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        H, W = frame.shape[:2]
        gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray  = cv2.equalizeHist(gray)   # helps in varied lighting

        # ── FPS ──────────────────────────────────────────────────────────────
        now = time.time()
        fps_deque.append(1.0 / max(now - t_prev, 1e-6))
        t_prev = now
        fps = np.mean(fps_deque)

        draw_panel(frame, W - 220, 0, 220, H)

        status_color = CLR_OK
        status_text  = "ALERT"
        ear_val      = 0.30   # neutral default (so gauge doesn't sit at 0)
        yawn_active  = False
        face_found   = False

        # ── face detection ────────────────────────────────────────────────────
        faces = FACE_CASCADE.detectMultiScale(
            gray, scaleFactor=1.1, minNeighbors=5,
            minSize=(120, 120), flags=cv2.CASCADE_SCALE_IMAGE
        )

        if len(faces) > 0:
            # use the largest detected face
            fx, fy, fw, fh = max(faces, key=lambda r: r[2] * r[3])
            face_found = True
            cv2.rectangle(frame, (fx, fy), (fx + fw, fy + fh), (80, 80, 80), 1)

            # ── head nod: track face centre-y ────────────────────────────────
            face_cy = fy + fh // 2
            face_y_history.append(face_cy / H)   # normalise
            nod_detected = False
            if len(face_y_history) == face_y_history.maxlen:
                baseline = np.mean(list(face_y_history)[:10])
                current  = np.mean(list(face_y_history)[-5:])
                if current - baseline > HEAD_NOD_THRESH:
                    nod_detected = True

            # ── ROI for eyes (top 55 % of face) ──────────────────────────────
            eye_roi_gray  = gray [fy: fy + int(fh * 0.55),
                                  fx: fx + fw]
            eye_roi_color = frame[fy: fy + int(fh * 0.55),
                                  fx: fx + fw]

            eyes = EYE_CASCADE.detectMultiScale(
                eye_roi_gray, scaleFactor=1.1, minNeighbors=6,
                minSize=(20, 20)
            )

            # ── ROI for mouth (bottom 45 % of face) ──────────────────────────
            mouth_y0 = fy + int(fh * 0.55)
            mouth_roi = gray[mouth_y0: fy + fh, fx: fx + fw]

            smiles = SMILE_CASCADE.detectMultiScale(
                mouth_roi, scaleFactor=1.7, minNeighbors=20,
                minSize=(25, 15)
            )
            yawn_active = len(smiles) > 0

            # draw mouth rect
            mouth_clr = CLR_WARN if yawn_active else (80, 80, 80)
            if len(smiles) > 0:
                sx, sy, sw, sh = smiles[0]
                cv2.rectangle(frame,
                              (fx + sx, mouth_y0 + sy),
                              (fx + sx + sw, mouth_y0 + sy + sh),
                              mouth_clr, 1)

            # ── per-eye EAR ───────────────────────────────────────────────────
            eye_ears = []
            for (ex, ey, ew, eh) in eyes[:2]:
                e = box_ear(ew, eh)
                eye_ears.append(e)
                clr = CLR_ALARM if e < EAR_THRESH else CLR_OK
                cv2.rectangle(eye_roi_color,
                              (ex, ey), (ex + ew, ey + eh), clr, 1)

            if eye_ears:
                ear_val = float(np.mean(eye_ears))
            else:
                # no eyes detected → treat as closed
                ear_val = 0.10

            eyes_closed = ear_val < EAR_THRESH

            # ── PERCLOS ───────────────────────────────────────────────────────
            perclos_buf.append(1 if eyes_closed else 0)
            perclos = np.mean(perclos_buf)

            # ── blink counter ─────────────────────────────────────────────────
            if eyes_closed and blink_open:
                blink_open = False
            elif not eyes_closed and not blink_open:
                session_blinks += 1
                blink_open = True

            # ── closure counter ───────────────────────────────────────────────
            if eyes_closed:
                ear_counter += 1
            else:
                ear_counter = 0

            # ── yawn counter ──────────────────────────────────────────────────
            if yawn_active:
                yawn_counter += 1
            else:
                if yawn_counter >= YAWN_CONSEC:
                    session_yawns += 1
                yawn_counter = 0

            # ── alarm logic ───────────────────────────────────────────────────
            drowsy = (ear_counter >= EAR_CONSEC_FRAMES or
                      perclos > PERCLOS_ALERT or
                      nod_detected)

            if drowsy:
                alarm_on     = True
                status_color = CLR_ALARM
                status_text  = "DROWSY!"
                play_alarm()
                alert_overlay(frame, "⚠  WAKE UP!  DROWSINESS DETECTED  ⚠", CLR_ALARM)
            elif yawn_active:
                alarm_on     = False
                status_color = CLR_WARN
                status_text  = "YAWNING"
                stop_alarm()
                alert_overlay(frame, "YAWNING DETECTED", CLR_WARN)
            else:
                alarm_on = False
                stop_alarm()

            # ── side-panel gauges ─────────────────────────────────────────────
            px = W - 205
            draw_gauge(frame, px, 40,  "EAR", ear_val, 0.0, 0.5, EAR_THRESH,
                       CLR_ALARM if eyes_closed else CLR_OK)
            draw_gauge(frame, px, 90,  "PERCLOS", perclos, 0.0, 1.0, PERCLOS_ALERT,
                       CLR_ALARM if perclos > PERCLOS_ALERT else CLR_OK)
            draw_gauge(frame, px, 140, "Closure fr", ear_counter / EAR_CONSEC_FRAMES,
                       0, 1, 0.9,
                       CLR_ALARM if ear_counter >= EAR_CONSEC_FRAMES else CLR_OK)

            put(frame, "NOD!" if nod_detected else "Head: stable",
                (px, 178), 0.40, CLR_WARN if nod_detected else CLR_TEXT)

        else:
            put(frame, "No face detected", (W - 210, 60), 0.50, CLR_WARN)
            perclos_buf.append(0)
            perclos = np.mean(perclos_buf)

        # ── stats & settings panel ────────────────────────────────────────────
        px = W - 205
        put(frame, "SESSION STATS",       (px, 210), 0.44, (160, 160, 160))
        put(frame, f"Yawns:  {session_yawns}",  (px, 230), 0.44)
        put(frame, f"Blinks: {session_blinks}", (px, 248), 0.44)
        put(frame, f"Closed: {ear_counter} fr", (px, 266), 0.44,
            CLR_ALARM if ear_counter > EAR_CONSEC_FRAMES // 2 else CLR_TEXT)

        put(frame, "SETTINGS",            (px, 296), 0.44, (160, 160, 160))
        put(frame, f"EAR thresh: {EAR_THRESH:.2f}", (px, 316), 0.44)
        put(frame, "+/- to adjust",       (px, 334), 0.38, (130, 130, 130))

        put(frame, f"FPS: {fps:.1f}",     (px, H - 50), 0.44)
        put(frame, "Q=quit R=reset S=save", (px, H - 30), 0.36, (130, 130, 130))

        # ── status badge ──────────────────────────────────────────────────────
        cv2.rectangle(frame, (0, 0), (200, 36), status_color, -1)
        put(frame, status_text, (8, 26), 0.80, (255, 255, 255), 2)

        cv2.imshow("Driver Drowsiness Detector", frame)
        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            print(f"\n[■] Session ended.  Yawns: {session_yawns}  Blinks: {session_blinks}")
            break
        elif key == ord("r"):
            ear_counter = yawn_counter = session_yawns = session_blinks = 0
            perclos_buf.clear(); face_y_history.clear()
            blink_open = True; alarm_on = False
            stop_alarm()
            print("[R] Counters reset.")
        elif key == ord("s"):
            fname = f"drowsiness_{datetime.now():%Y%m%d_%H%M%S}.jpg"
            cv2.imwrite(fname, frame)
            print(f"[📸] Snapshot saved: {fname}")
        elif key in (ord("+"), ord("=")):
            EAR_THRESH = min(EAR_THRESH + 0.01, 0.40)
            print(f"[+] EAR threshold → {EAR_THRESH:.2f}  (less sensitive)")
        elif key == ord("-"):
            EAR_THRESH = max(EAR_THRESH - 0.01, 0.10)
            print(f"[-] EAR threshold → {EAR_THRESH:.2f}  (more sensitive)")

    cap.release()
    cv2.destroyAllWindows()
    if AUDIO_OK:
        pygame.mixer.quit()


if __name__ == "__main__":
    main()

pygame 2.6.1 (SDL 2.28.4, Python 3.9.13)
Hello from the pygame community. https://www.pygame.org/contribute.html

[▶] Drowsiness Detector running (OpenCV cascade edition)
    Q=quit  R=reset  S=save snapshot  +/-=sensitivity


[■] Session ended.  Yawns: 14  Blinks: 49
